In [6]:
from pathlib import Path
import pandas as pd


def consolidar_iniciativas_inversion(
    carpeta_archivos: str, output_excel: str = None
) -> pd.DataFrame:
  """Lee todos los archivos Excel de una carpeta y consolida las iniciativas de inversión.

  Parámetros:
  -----------
  carpeta_archivos : str o Path
      Ruta a la carpeta donde están los archivos Excel.
  output_excel : str, opcional
      Ruta/nombre de archivo si deseas guardar el resultado consolidado en un
      nuevo Excel.
  """
  ruta = Path(carpeta_archivos)
  archivos_excel = list(ruta.glob("*.xlsx")) + list(ruta.glob("*.xls"))

  if not archivos_excel:
    print(f"No se encontraron archivos Excel en: {ruta.resolve()}")
    return pd.DataFrame()

  print(f"Se encontraron {len(archivos_excel)} archivos. Procesando...")

  lista_dfs = []
  columnas_requeridas = [
      "CÓDIGO BIP",
      "SOLICITADO AÑO",
      "COSTO TOTAL [M$]",
      "ETAPA ACTUAL",
      "AÑO POSTULACIÓN",
  ]

  for archivo in archivos_excel:
    # Ignorar archivos temporales abiertos por Excel (que empiezan con ~$)
    if archivo.name.startswith("~$"):
      continue

    try:
      # Leer el archivo Excel
      df_temp = pd.read_excel(archivo)

      # Limpiar espacios en blanco en los nombres de columnas
      df_temp.columns = df_temp.columns.astype(str).str.strip()

      # Validar que contenga las columnas necesarias
      cols_presentes = [c for c in columnas_requeridas if c in df_temp.columns]
      if len(cols_presentes) == len(columnas_requeridas):
        lista_dfs.append(df_temp[columnas_requeridas])
      else:
        faltantes = set(columnas_requeridas) - set(df_temp.columns)
        print(f"Advertencia: El archivo '{archivo.name}' no tiene: {faltantes}")

    except Exception as e:
      print(f"Error al leer '{archivo.name}': {e}")

  if not lista_dfs:
    print("No se pudieron extraer datos válidos de los archivos.")
    return pd.DataFrame()

  # 1. Unir todos los DataFrames
  df_total = pd.concat(lista_dfs, ignore_index=True)

  # 2. Limpieza y conversión de tipos de datos
  # Asegurar que el año sea numérico para ordenar correctamente
  df_total["AÑO POSTULACIÓN"] = pd.to_numeric(
      df_total["AÑO POSTULACIÓN"], errors="coerce"
  )
  df_total["SOLICITADO AÑO"] = (
      pd.to_numeric(df_total["SOLICITADO AÑO"], errors="coerce").fillna(0)
  )

  # Eliminar filas donde el código o el año sean nulos
  df_total = df_total.dropna(subset=["CÓDIGO BIP", "AÑO POSTULACIÓN"])

  # 3. Ordenar por Código y Año ascendente
  df_total = df_total.sort_values(
      by=["CÓDIGO BIP", "AÑO POSTULACIÓN"], ascending=[True, True]
  )

  # 4. Agrupar y resumir por proyecto
  df_resumen = (
      df_total.groupby("CÓDIGO BIP", as_index=False)
      .agg(
          **{
              "Total Solicitado": ("SOLICITADO AÑO", "sum"),
              "Ultimo Costo Total": ("COSTO TOTAL [M$]", "last"),
              "Ultimo Estado": ("ETAPA ACTUAL", "last"),
              "Año Primera Postulación": ("AÑO POSTULACIÓN", "first"),
              "Año Ultima Postulación": ("AÑO POSTULACIÓN", "last"),
          }
      )
  )

  # Convertir años a enteros para evitar formato float (ej. 2021.0 -> 2021)
  df_resumen["Año Primera Postulación"] = df_resumen[
      "Año Primera Postulación"
  ].astype(int)
  df_resumen["Año Ultima Postulación"] = df_resumen[
      "Año Ultima Postulación"
  ].astype(int)

  print(
      f"¡Consolidación exitosa! {len(df_resumen)} proyectos únicos procesados."
  )

  # 5. Opcional: Guardar a Excel
  if output_excel:
    df_resumen.to_excel(output_excel, index=False)
    print(f"Resultado guardado en: {output_excel}")

  return df_resumen




In [7]:
# ==========================================
# EJEMPLO DE USO:
# ==========================================
if __name__ == "__main__":
  # Cambia esta ruta por la carpeta donde tienes tus archivos
  CARPETA_EXCEL = "C:\\Users\\Edison\\Desktop\\CATLEC\\BIP"

  # Ejecutar consolidación
  df_final = consolidar_iniciativas_inversion(
      carpeta_archivos=CARPETA_EXCEL,
      output_excel="Resumen_Iniciativas_Consolidado.xlsx",
  )

  # Mostrar las primeras filas
  print(df_final.head())

Se encontraron 19 archivos. Procesando...
¡Consolidación exitosa! 190797 proyectos únicos procesados.
Resultado guardado en: Resumen_Iniciativas_Consolidado.xlsx
   CÓDIGO BIP  Total Solicitado  Ultimo Costo Total Ultimo Estado  \
0      100018               230                 230     EJECUCION   
1      100685             60000               60000     EJECUCION   
2      101076            907514             5160484     EJECUCION   
3      300597            170000            23391600     EJECUCION   
4      400098            766525              949667     EJECUCION   

   Año Primera Postulación  Año Ultima Postulación  
0                     1997                    1997  
1                     1999                    1999  
2                     1997                    1998  
3                     1997                    1997  
4                     1997                    1998  
